# Setwise distillation pilot

The teacher compares **k documents at once** and names a winner. Those choices are the training labels. The student scores **one document at a time**, trained with a setwise loss, and at inference the list is sorted by that score.

This run is a small pilot on Jusbrasil:

- **Train:** 50 queries, all of their candidate documents. The teacher labels these.
- **Validation:** 50 other queries, used only to measure nDCG. The teacher is not called.
- **Test:** every query in `small_test.parquet`.

Human `relevance` is only for nDCG. The training target is the teacher winner.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.config import DistillConfig
from src.data.schema import load_examples, save_examples
from src.models.teacher import SetwiseTeacher

## Data

`train.parquet` and `vali.parquet` are cut down to 50 queries each. Every candidate for a chosen query stays in the frame. `small_test.parquet` is used whole.

In [ ]:
cfg = DistillConfig(
    dataset_name="jusbrasil",
    teacher_client="ollama",
    teacher_model="llama3.1:8b",
    k_size=4,
    sets_per_query=2,
    max_doc_chars=1500,
)

DATA_DIR = Path("/Users/david/Documents/data/jusbrasil/manual/with_body/splits")
N_PILOT_QUERIES = 50


def take_queries(df, n_queries, seed):
    """Keep every document from n_queries distinct queries."""
    queries = pd.Series(df["query"].unique())
    if len(queries) <= n_queries:
        return df.copy()
    chosen = set(queries.sample(n=n_queries, random_state=seed))
    return df[df["query"].isin(chosen)].copy()


train_all = pd.read_parquet(DATA_DIR / "train.parquet")
val_all = pd.read_parquet(DATA_DIR / "vali.parquet")
test_df = pd.read_parquet(DATA_DIR / "small_test.parquet")

train_df = take_queries(train_all, N_PILOT_QUERIES, cfg.seed)
val_df = take_queries(val_all, N_PILOT_QUERIES, cfg.seed)


def describe(name, frame):
    sizes = frame.groupby("query").size()
    ready = int((sizes >= cfg.k_size).sum())
    print(
        f"{name}: {frame['query'].nunique()} queries, {len(frame)} rows, "
        f"{ready} queries with at least {cfg.k_size} docs"
    )


describe("train", train_df)
describe("val", val_df)
describe("test", test_df)

## 1. Teacher

One comparison on the first pilot query, before labeling all 50. The reply should be an index into the set, or `-1` when the letter cannot be parsed.

In [ ]:
teacher = SetwiseTeacher(
    api_client_type=cfg.teacher_client,
    model_name=cfg.teacher_model,
    k_size=cfg.k_size,
    dataset_name=cfg.dataset_name,
    max_chars=cfg.max_doc_chars,
)

query_text = train_df["query"].iloc[0]
group = train_df[train_df["query"] == query_text]
docs = group.head(cfg.k_size).to_dict(orient="records")
print(query_text)
print(f"{len(group)} documents for this query; sending {len(docs)} to the teacher")
for index, doc in enumerate(docs):
    preview = teacher.format_doc(doc)[:120].replace("\n", " | ")
    print(index, preview)
print("winner index:", teacher.compare_documents_setwise(docs, query_text))

## 2. Labels

`from_samples` draws `sets_per_query` subsets of size `k` from each of the 50 training queries and asks the teacher which document wins. Queries with fewer than `k` documents are skipped.

`from_rank_trace` is the alternative: run the heapsort and keep every comparison it made.

In [ ]:
from src.engine.generator import SetwiseLabelGenerator

generator = SetwiseLabelGenerator(
    teacher,
    sets_per_query=cfg.sets_per_query,
    seed=cfg.seed,
)
generated = generator.from_samples(train_df)
print(len(generated.examples), "examples,", generated.skipped, "skipped")
print(generated.queries_seen, "train queries seen")
generated.examples[0]

In [ ]:
LABELS_PATH = ROOT / "notebooks" / "data" / "setwise_labels.jsonl"
save_examples(generated.examples, LABELS_PATH)
examples = load_examples(LABELS_PATH)
len(examples)

## 3. Setwise loss

Implement `setwise_loss`. For each set, the student scores are logits and the teacher winner is the class. Average the cross-entropy over the batch.

Do not use this function at inference. Inference sorts by the raw score.

`listmle_loss` and `pairwise_ranknet_loss` are optional. Leave them until the setwise path trains.

In [ ]:
import torch

from src.loss.setwise import setwise_loss

scores = torch.tensor([[2.0, 0.0, 0.0]])
winner = torch.tensor([0])
value = float(setwise_loss(scores, winner))
print(value)
assert abs(value - 0.2395) < 1e-3

## 4. Student

Implement a pointwise scorer in `StudentRanker`. `score_sets` is already written: it flattens each set, calls `score_pairs`, and reshapes to `[batch, k]`.

A cross-encoder is the usual model. Load it in `__init__` and return one logit per pair from `score_pairs`. Higher means more relevant. Install Transformers with `poetry add transformers` when you get there.

The trainer raises a clear error if the module has no parameters.

Each epoch prints the training loss and nDCG@10 on the 50 training queries, and appends that row to `notebooks/data/train_log.csv`. After the run:

```bash
poetry run python scripts/plot_learning_curve.py notebooks/data/train_log.csv
```

`SANITY_EPOCHS` is set above 1 so the curve can fall. The loss is the setwise loss. nDCG@10 uses the human grades on the same queries.

In [ ]:
from src.engine.trainer import SetwiseLoader, SetwiseTrainer
from src.models.student import StudentRanker

LOG_PATH = ROOT / "notebooks" / "data" / "train_log.csv"
SANITY_EPOCHS = 5

student = StudentRanker(cfg.student_model)
loader = SetwiseLoader(examples, batch_size=cfg.batch_size, shuffle=True, seed=cfg.seed)
trainer = SetwiseTrainer(student, setwise_loss, lr=cfg.learning_rate)
history = trainer.fit(
    loader,
    epochs=SANITY_EPOCHS,
    log_path=LOG_PATH,
    ndcg_frame=train_df,
    format_fn=teacher.format_doc,
)
history

## 5. Pointwise inference

Each candidate is scored alone with `teacher.format_doc`, then sorted. nDCG uses the human grade.

Validation is the 50-query slice. Test is the whole `small_test.parquet`.

In [ ]:
from src.engine.evaluator import evaluate_pointwise

val_metrics = evaluate_pointwise(student, val_df, format_fn=teacher.format_doc)
test_metrics = evaluate_pointwise(student, test_df, format_fn=teacher.format_doc)
print("val ", val_metrics.means)
print("test", test_metrics.means)